# Section 1: Project Setup and Environment Configuration

## Introduction

This notebook implements the complete deep learning training pipeline for automated skin lesion classification using the enhanced HAM10000 dataset. The objective is to train a robust convolutional neural network capable of accurately classifying seven different categories of pigmented skin lesions.

The enhanced images generated during the preprocessing stage are used as input to improve image quality before model training. The notebook follows a structured workflow beginning with project configuration, dataset preparation, model development, transfer learning, fine-tuning, and performance evaluation.

A reproducible environment is established by defining project directories, random seeds, and training parameters before loading the dataset.

---

## Objectives

- Configure the project environment.
- Define all project directories.
- Initialize global training parameters.
- Ensure reproducibility using fixed random seeds.
- Verify the availability of required folders.

---

## Expected Output

After completing this section:

- Project directories will be initialized.
- Global constants will be created.
- Random seeds will be fixed.
- The notebook environment will be ready for dataset loading.

In [12]:
from pathlib import Path
import os
import random
import warnings

import numpy as np
import pandas as pd
import tensorflow as tf

warnings.filterwarnings("ignore")

In [13]:
PROJECT_DIR = Path(r"D:\Projects\DermaCQI")

DATASET_DIR = PROJECT_DIR / "datasets"
OUTPUT_DIR = PROJECT_DIR / "outputs"
MODEL_DIR = PROJECT_DIR / "models"
LOG_DIR = PROJECT_DIR / "logs"

ENHANCED_IMAGE_DIR = OUTPUT_DIR / "Enhanced_HAM10000"

METADATA_PATH = DATASET_DIR / "HAM10000_metadata.csv"

In [14]:
IMAGE_SIZE = (224, 224)

BATCH_SIZE = 32

NUM_CLASSES = 7

RANDOM_STATE = 42

AUTOTUNE = tf.data.AUTOTUNE

In [15]:
random.seed(RANDOM_STATE)

np.random.seed(RANDOM_STATE)

tf.random.set_seed(RANDOM_STATE)

In [16]:
for folder in [
    PROJECT_DIR,
    DATASET_DIR,
    OUTPUT_DIR,
    MODEL_DIR,
    LOG_DIR,
    ENHANCED_IMAGE_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

In [17]:
print("Project Directory :", PROJECT_DIR)
print("Dataset Directory :", DATASET_DIR)
print("Enhanced Images   :", ENHANCED_IMAGE_DIR)
print("Metadata File     :", METADATA_PATH)
print("Model Directory   :", MODEL_DIR)
print("Logs Directory    :", LOG_DIR)

Project Directory : D:\Projects\DermaCQI
Dataset Directory : D:\Projects\DermaCQI\datasets
Enhanced Images   : D:\Projects\DermaCQI\outputs\Enhanced_HAM10000
Metadata File     : D:\Projects\DermaCQI\datasets\HAM10000_metadata.csv
Model Directory   : D:\Projects\DermaCQI\models
Logs Directory    : D:\Projects\DermaCQI\logs


In [18]:
print("TensorFlow Version :", tf.__version__)
print("NumPy Version      :", np.__version__)
print("Pandas Version     :", pd.__version__)

TensorFlow Version : 2.21.0
NumPy Version      : 2.4.6
Pandas Version     : 3.0.3


In [19]:
gpus = tf.config.list_physical_devices("GPU")

if gpus:
    print("GPU Detected")
    print(gpus[0].name)
else:
    print("Running on CPU")

Running on CPU


# Section 2: Load and Validate Metadata

## Introduction

This section loads the HAM10000 metadata file and validates its contents. The metadata contains the image names and disease labels that will be used throughout the training process.

The image paths are created by linking every image ID with the enhanced image folder. Basic validation is performed to ensure that the metadata is complete before training begins.

---

## Objectives

- Load the metadata file.
- Create image paths.
- Check for missing values.
- Verify image files.
- Display dataset information.

---

## Expected Output

After completing this section:

- Metadata will be loaded successfully.
- Image paths will be created.
- Missing values will be checked.
- Dataset information will be displayed.

In [20]:
# Load metadata

metadata = pd.read_csv(METADATA_PATH)

metadata.head()

,lesion_id,image_id,dx,dx_type,age,sex,localization
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear


In [21]:
# Create image path

metadata["image_path"] = metadata["image_id"].apply(
    lambda x: str(ENHANCED_IMAGE_DIR / f"{x}.jpg")
)

metadata.head()

,lesion_id,image_id,dx,dx_type,age,sex,localization,image_path
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp,D:\Projects\DermaCQI\outputs\Enhanced_HAM10000...
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp,D:\Projects\DermaCQI\outputs\Enhanced_HAM10000...
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp,D:\Projects\DermaCQI\outputs\Enhanced_HAM10000...
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp,D:\Projects\DermaCQI\outputs\Enhanced_HAM10000...
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear,D:\Projects\DermaCQI\outputs\Enhanced_HAM10000...


In [22]:
# Check dataset shape

print("Rows :", metadata.shape[0])
print("Columns :", metadata.shape[1])

Rows : 10015
Columns : 8


In [23]:
# Check missing values

metadata.isnull().sum()

lesion_id        0
image_id         0
dx               0
dx_type          0
age             57
sex              0
localization     0
image_path       0
dtype: int64

In [24]:
# Check duplicate rows

print("Duplicate Rows :", metadata.duplicated().sum())

Duplicate Rows : 0


In [25]:
# Verify image files

metadata["image_exists"] = metadata["image_path"].apply(
    lambda x: Path(x).exists()
)

print(metadata["image_exists"].value_counts())

image_exists
True    10015
Name: count, dtype: int64


In [26]:
# Keep only available images

metadata = metadata[
    metadata["image_exists"] == True
].copy()

metadata.reset_index(drop=True, inplace=True)

print("Available Images :", len(metadata))

Available Images : 10015


In [27]:
# Show dataset information

metadata.info()

<class 'pandas.DataFrame'>
RangeIndex: 10015 entries, 0 to 10014
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   lesion_id     10015 non-null  str    
 1   image_id      10015 non-null  str    
 2   dx            10015 non-null  str    
 3   dx_type       10015 non-null  str    
 4   age           9958 non-null   float64
 5   sex           10015 non-null  str    
 6   localization  10015 non-null  str    
 7   image_path    10015 non-null  str    
 8   image_exists  10015 non-null  bool   
dtypes: bool(1), float64(1), str(7)
memory usage: 635.8 KB


In [28]:
# Display random samples

metadata.sample(5)

,lesion_id,image_id,dx,dx_type,age,sex,localization,image_path,image_exists
1617,HAM_0007180,ISIC_0033272,mel,histo,65.0,male,face,D:\Projects\DermaCQI\outputs\Enhanced_HAM10000...,True
8128,HAM_0007195,ISIC_0031923,nv,histo,40.0,female,lower extremity,D:\Projects\DermaCQI\outputs\Enhanced_HAM10000...,True
2168,HAM_0001835,ISIC_0026652,mel,histo,65.0,male,back,D:\Projects\DermaCQI\outputs\Enhanced_HAM10000...,True
1090,HAM_0000465,ISIC_0030583,bkl,consensus,35.0,female,trunk,D:\Projects\DermaCQI\outputs\Enhanced_HAM10000...,True
7754,HAM_0001720,ISIC_0034010,nv,histo,45.0,male,abdomen,D:\Projects\DermaCQI\outputs\Enhanced_HAM10000...,True


# Section 3: Label Encoding and Dataset Balancing

## Introduction

This section prepares the target labels for model training. The disease labels are converted into numerical values that can be used by the deep learning model.

Since the HAM10000 dataset is highly imbalanced, class weights are calculated to give higher importance to minority classes during training. This allows the model to learn from all classes without removing any images.

---

## Objectives

- Encode disease labels.
- Create label mapping.
- Calculate class weights.
- Prepare labels for training.

---

## Expected Output

After completing this section:

- Disease labels will be encoded.
- Label mapping will be created.
- Class weights will be calculated.
- Dataset will be ready for train-test splitting.

In [29]:
# Import libraries

from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

In [30]:
# Encode labels

label_encoder = LabelEncoder()

metadata["label"] = label_encoder.fit_transform(metadata["dx"])

In [31]:
# Create label mapping

label_mapping = dict(
    zip(
        label_encoder.classes_,
        label_encoder.transform(label_encoder.classes_)
    )
)

label_mapping

{'akiec': np.int64(0),
 'bcc': np.int64(1),
 'bkl': np.int64(2),
 'df': np.int64(3),
 'mel': np.int64(4),
 'nv': np.int64(5),
 'vasc': np.int64(6)}

In [32]:
# Show class names

class_names = list(label_encoder.classes_)

class_names

['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']

In [33]:
# Count samples in each class

metadata["dx"].value_counts()

dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64

In [34]:
# Calculate class weights

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(metadata["label"]),
    y=metadata["label"]
)

class_weights = dict(enumerate(weights))

class_weights

{0: np.float64(4.375273044997815),
 1: np.float64(2.78349082823791),
 2: np.float64(1.301832835044846),
 3: np.float64(12.440993788819876),
 4: np.float64(1.2854575792581184),
 5: np.float64(0.21338020666879728),
 6: np.float64(10.075452716297788)}

In [35]:
# Show encoded dataset

metadata.head()

,lesion_id,image_id,dx,dx_type,age,sex,localization,image_path,image_exists,label
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp,D:\Projects\DermaCQI\outputs\Enhanced_HAM10000...,True,2
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp,D:\Projects\DermaCQI\outputs\Enhanced_HAM10000...,True,2
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp,D:\Projects\DermaCQI\outputs\Enhanced_HAM10000...,True,2
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp,D:\Projects\DermaCQI\outputs\Enhanced_HAM10000...,True,2
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear,D:\Projects\DermaCQI\outputs\Enhanced_HAM10000...,True,2


In [36]:
# Check number of classes

print("Number of Classes :", len(class_names))

Number of Classes : 7


In [37]:
# Display class names

for index, name in enumerate(class_names):
    print(index, "-", name)

0 - akiec
1 - bcc
2 - bkl
3 - df
4 - mel
5 - nv
6 - vasc


# Section 4: Train, Validation and Test Split

## Introduction

This section divides the dataset into training, validation, and testing sets. A stratified split is used so that each set maintains a similar distribution of disease classes.

The training set is used for learning, the validation set is used for monitoring model performance during training, and the test set is reserved for the final evaluation.

---

## Objectives

- Split the dataset.
- Preserve class distribution.
- Create training, validation and test sets.
- Verify the split.

---

## Expected Output

After completing this section:

- Training dataset will be created.
- Validation dataset will be created.
- Test dataset will be created.
- Class distribution will be maintained.

In [38]:
# Import library

from sklearn.model_selection import train_test_split

In [39]:
# Split train and test

train_df, test_df = train_test_split(
    metadata,
    test_size=0.15,
    stratify=metadata["label"],
    random_state=RANDOM_STATE
)

In [40]:
# Split train and validation

train_df, val_df = train_test_split(
    train_df,
    test_size=0.15,
    stratify=train_df["label"],
    random_state=RANDOM_STATE
)

In [41]:
# Reset index

train_df = train_df.reset_index(drop=True)

val_df = val_df.reset_index(drop=True)

test_df = test_df.reset_index(drop=True)

In [42]:
# Show dataset size

print("Training Images :", len(train_df))

print("Validation Images :", len(val_df))

print("Testing Images :", len(test_df))

Training Images : 7235
Validation Images : 1277
Testing Images : 1503


In [43]:
# Show training distribution

train_df["dx"].value_counts()

dx
nv       4844
mel       804
bkl       794
bcc       372
akiec     236
vasc      102
df         83
Name: count, dtype: int64

In [44]:
# Show validation distribution

val_df["dx"].value_counts()

dx
nv       855
mel      142
bkl      140
bcc       65
akiec     42
vasc      18
df        15
Name: count, dtype: int64

In [45]:
# Show testing distribution

test_df["dx"].value_counts()

dx
nv       1006
mel       167
bkl       165
bcc        77
akiec      49
vasc       22
df         17
Name: count, dtype: int64

In [46]:
# Display random training samples

train_df.sample(5)

,lesion_id,image_id,dx,dx_type,age,sex,localization,image_path,image_exists,label
2697,HAM_0007587,ISIC_0029762,nv,follow_up,60.0,female,lower extremity,D:\Projects\DermaCQI\outputs\Enhanced_HAM10000...,True,5
3500,HAM_0000631,ISIC_0030885,nv,follow_up,45.0,female,trunk,D:\Projects\DermaCQI\outputs\Enhanced_HAM10000...,True,5
247,HAM_0003816,ISIC_0029152,nv,consensus,40.0,male,unknown,D:\Projects\DermaCQI\outputs\Enhanced_HAM10000...,True,5
3256,HAM_0002734,ISIC_0028190,akiec,histo,45.0,male,face,D:\Projects\DermaCQI\outputs\Enhanced_HAM10000...,True,0
3277,HAM_0000949,ISIC_0032444,nv,histo,65.0,male,back,D:\Projects\DermaCQI\outputs\Enhanced_HAM10000...,True,5


In [47]:
# Save split files

train_df.to_csv(OUTPUT_DIR / "train.csv", index=False)

val_df.to_csv(OUTPUT_DIR / "validation.csv", index=False)

test_df.to_csv(OUTPUT_DIR / "test.csv", index=False)

In [48]:
# Confirm files are saved

print("Train CSV :", OUTPUT_DIR / "train.csv")

print("Validation CSV :", OUTPUT_DIR / "validation.csv")

print("Test CSV :", OUTPUT_DIR / "test.csv")

Train CSV : D:\Projects\DermaCQI\outputs\train.csv
Validation CSV : D:\Projects\DermaCQI\outputs\validation.csv
Test CSV : D:\Projects\DermaCQI\outputs\test.csv


# Section 5: Build TensorFlow Data Pipeline

## Introduction

This section creates the TensorFlow data pipeline for loading and preprocessing images. The pipeline reads images directly from disk, resizes them, normalizes pixel values, and prepares batches for training.

Data augmentation is also applied to the training dataset to improve the model's ability to generalize and reduce overfitting.

---

## Objectives

- Load images.
- Apply preprocessing.
- Apply data augmentation.
- Create TensorFlow datasets.
- Improve training performance.

---

## Expected Output

After completing this section:

- Training dataset will be created.
- Validation dataset will be created.
- Test dataset will be created.
- Data augmentation will be ready.

In [52]:
import tensorflow as tf

from tensorflow.keras import layers
from tensorflow.keras import models
from tensorflow.keras import callbacks

In [53]:
# Create augmentation layer

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.05)
])

In [54]:
def load_image(image_path, label):

    image = tf.io.read_file(image_path)

    image = tf.image.decode_jpeg(image, channels=3)

    image = tf.image.resize(image, IMAGE_SIZE)

    image = tf.cast(image, tf.float32)

    image = tf.keras.applications.efficientnet_v2.preprocess_input(image)

    return image, label

In [55]:
# Apply augmentation

def augment_image(image, label):

    image = data_augmentation(image)

    return image, label

In [56]:
# Create TensorFlow dataset

def create_dataset(dataframe, training=False):

    dataset = tf.data.Dataset.from_tensor_slices(
        (
            dataframe["image_path"].values,
            dataframe["label"].values
        )
    )

    dataset = dataset.map(
        load_image,
        num_parallel_calls=AUTOTUNE
    )

    if training:

        dataset = dataset.shuffle(
            buffer_size=len(dataframe),
            seed=RANDOM_STATE
        )

        dataset = dataset.map(
            augment_image,
            num_parallel_calls=AUTOTUNE
        )

    dataset = dataset.batch(BATCH_SIZE)

    dataset = dataset.prefetch(AUTOTUNE)

    return dataset

In [57]:
# Create datasets

train_ds = create_dataset(train_df, training=True)

val_ds = create_dataset(val_df)

test_ds = create_dataset(test_df)

In [58]:
# Check dataset

print(train_ds)

print(val_ds)

print(test_ds)

<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))>
<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))>
<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))>


In [59]:
# Show dataset size

print("Training Batches :", tf.data.experimental.cardinality(train_ds).numpy())

print("Validation Batches :", tf.data.experimental.cardinality(val_ds).numpy())

print("Testing Batches :", tf.data.experimental.cardinality(test_ds).numpy())

Training Batches : 227
Validation Batches : 40
Testing Batches : 47


In [60]:
# Display one batch

images, labels = next(iter(train_ds))

print(images.shape)

print(labels.shape)

(32, 224, 224, 3)
(32,)


In [61]:
# Show image range

print(images.dtype)

print(tf.reduce_min(images).numpy())

print(tf.reduce_max(images).numpy())

<dtype: 'float32'>
0.00092514104
255.0


In [62]:
# Show sample labels

print(labels.numpy())

[5 5 5 5 5 5 5 0 4 5 1 5 5 5 5 5 1 2 5 5 5 5 5 5 5 4 5 5 5 2 0 5]


# Section 6: Build EfficientNetV2B0 Model

## Introduction

This section builds the deep learning model using EfficientNetV2B0 with ImageNet pretrained weights. The pretrained backbone is used as a feature extractor, while a custom classification head is added to classify the seven skin lesion categories.

During the first stage of training, the backbone remains frozen so that only the new classification layers are trained.

---

## Objectives

- Load EfficientNetV2B0.
- Use ImageNet pretrained weights.
- Build the classification head.
- Freeze the backbone.
- Compile the model.

---

## Expected Output

After completing this section:

- EfficientNetV2B0 model will be created.
- The backbone will remain frozen.
- The model will be compiled.
- The model will be ready for Stage 1 training.

In [63]:
# Import libraries

from tensorflow.keras import layers
from tensorflow.keras import models
from tensorflow.keras import applications

In [64]:
# Load pretrained model

base_model = applications.EfficientNetV2B0(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze backbone

base_model.trainable = False

In [65]:
# Build model

inputs = layers.Input(shape=(224, 224, 3))

x = base_model(inputs, training=False)

x = layers.GlobalAveragePooling2D()(x)

x = layers.BatchNormalization()(x)

x = layers.Dropout(0.3)(x)

outputs = layers.Dense(
    NUM_CLASSES,
    activation="softmax"
)(x)

model = models.Model(inputs, outputs)

In [66]:
# Compile model

model.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-3
    ),

    loss="sparse_categorical_crossentropy",

    metrics=[
        "accuracy"
    ]
)

In [67]:
# Show model summary

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetv2-b0 (Functional)  │ (None, 7, 7, 1280)     │     5,919,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 7)              │         8,967 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,933,399 (22.63 MB)

 Trainable params: 11,527 (45.03 KB)

 Non-trainable params: 5,921,872 (22.59 MB)

In [68]:
# Count parameters

print("Total Parameters :", model.count_params())

Total Parameters : 5933399


In [69]:
# Check trainable layers

print("Trainable Variables :", len(model.trainable_variables))

print("Non Trainable Variables :", len(model.non_trainable_variables))

Trainable Variables : 4
Non Trainable Variables : 377


In [70]:
# Check backbone

print(base_model.trainable)

False


In [71]:
# Save model architecture

tf.keras.utils.plot_model(
    model,
    show_shapes=True,
    show_dtype=False,
    expand_nested=True
)

You must install graphviz (see instructions at https://graphviz.gitlab.io/download/) for `plot_model` to work.


In [72]:
# Display model

model

<Functional name=functional_1, built=True>

# Section 7: Stage 1 Model Training

## Introduction

This section performs the first stage of transfer learning. During this stage, the EfficientNetV2B0 backbone remains frozen while only the classification head is trained.

Several callbacks are used to improve training stability and prevent overfitting. The model with the best validation accuracy is automatically saved for later fine-tuning.

---

## Objectives

- Configure training callbacks.
- Train the classification head.
- Save the best model.
- Monitor validation performance.

---

## Expected Output

After completing this section:

- Stage 1 training will be completed.
- Best model will be saved.
- Training history will be recorded.
- Validation accuracy and loss will be monitored.

In [73]:
# Import callbacks

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau,
    CSVLogger
)

In [74]:
# Create callbacks

checkpoint = ModelCheckpoint(
    filepath=MODEL_DIR / "efficientnetv2b0_stage1_best.keras",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

csv_logger = CSVLogger(
    LOG_DIR / "stage1_training_log.csv"
)

In [75]:
# Store callbacks

callbacks = [
    checkpoint,
    early_stopping,
    reduce_lr,
    csv_logger
]

In [214]:
# Start training

history = model.fit(

    train_ds,

    validation_data=val_ds,

    epochs=20,

    class_weight=class_weights,

    callbacks=callbacks,

    verbose=1
)

Epoch 1/20
227/227 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.3397 - loss: 1.9739
Epoch 1: val_accuracy improved from None to 0.60846, saving model to D:\Projects\DermaCQI\models\efficientnetv2b0_stage1_best.keras

Epoch 1: finished saving model to D:\Projects\DermaCQI\models\efficientnetv2b0_stage1_best.keras
227/227 ━━━━━━━━━━━━━━━━━━━━ 325s 1s/step - accuracy: 0.3397 - loss: 1.9739 - val_accuracy: 0.6085 - val_loss: 1.1294 - learning_rate: 0.0010
Epoch 2/20
227/227 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.4756 - loss: 1.5031
Epoch 2: val_accuracy did not improve from 0.60846
227/227 ━━━━━━━━━━━━━━━━━━━━ 315s 1s/step - accuracy: 0.4756 - loss: 1.5031 - val_accuracy: 0.5959 - val_loss: 1.0999 - learning_rate: 0.0010
Epoch 3/20
227/227 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.5212 - loss: 1.3610
Epoch 3: val_accuracy improved from 0.60846 to 0.60924, saving model to D:\Projects\DermaCQI\models\efficientnetv2b0_stage1_best.keras

Epoch 3: finished saving model to D:\Project

In [76]:
# Save final model

model.save(
    MODEL_DIR / "efficientnetv2b0_stage1_final.keras"
)

In [88]:
# Show history keys

history.history.keys()

dict_keys(['accuracy', 'loss', 'val_accuracy', 'val_loss', 'learning_rate'])

In [89]:
# Best validation accuracy

print(
    max(history.history["val_accuracy"])
)

0.6640563607215881


In [90]:
# Best validation loss

print(
    min(history.history["val_loss"])
)

0.951909065246582


In [91]:
# Final training accuracy

print(
    history.history["accuracy"][-1]
)

0.6092605590820312


In [92]:
# Final training loss

print(
    history.history["loss"][-1]
)

0.9587541818618774


In [93]:
# Best validation accuracy
print(max(history.history["val_accuracy"]))

# Best validation loss
print(min(history.history["val_loss"]))

# Final training accuracy
print(history.history["accuracy"][-1])

# Final training loss
print(history.history["loss"][-1])

0.6640563607215881
0.951909065246582
0.6092605590820312
0.9587541818618774


In [94]:
# Save final model

model.save(
    MODEL_DIR / "efficientnetv2b0_stage1_final.keras"
)

In [95]:
# Stage 1 Summary

print(f"Best Validation Accuracy : {max(history.history['val_accuracy']):.4f}")
print(f"Best Validation Loss     : {min(history.history['val_loss']):.4f}")
print(f"Final Training Accuracy  : {history.history['accuracy'][-1]:.4f}")
print(f"Final Training Loss      : {history.history['loss'][-1]:.4f}")

Best Validation Accuracy : 0.6641
Best Validation Loss     : 0.9519
Final Training Accuracy  : 0.6093
Final Training Loss      : 0.9588


In [96]:
import pandas as pd

# Load Stage 1 log
log_df = pd.read_csv(r"D:\Projects\DermaCQI\logs\stage1_training_log.csv")

# Recreate history dictionary
history = type("History", (), {})()

history.history = {
    "accuracy": log_df["accuracy"].tolist(),
    "loss": log_df["loss"].tolist(),
    "val_accuracy": log_df["val_accuracy"].tolist(),
    "val_loss": log_df["val_loss"].tolist(),
    "learning_rate": log_df["learning_rate"].tolist()
}

print(history.history.keys())

dict_keys(['accuracy', 'loss', 'val_accuracy', 'val_loss', 'learning_rate'])


In [97]:
print(max(history.history["val_accuracy"]))
print(min(history.history["val_loss"]))
print(history.history["accuracy"][-1])
print(history.history["loss"][-1])

0.6640563607215881
0.951909065246582
0.6092605590820312
0.9587541818618774
